# Silver Layer Data Processing

## Purpose
This notebook transforms raw sales data from the **bronze layer** into clean, validated data in the **silver layer**.

## Process Flow
1. **Read**: Load data from `sales.bronze.retail_store_sales`
2. **Clean**: Remove invalid rows, standardize data types
3. **Impute**: Fill missing sales metrics using business logic
4. **Validate**: Filter out remaining incomplete records
5. **Write**: Save to `sales.silver.retail_store_sales` as Delta table

## Input
- Delta table: `sales.bronze.retail_store_sales`

## Output
- Delta table: `sales.siver.retail_store_sales` with clean data

In [0]:
# ============================================================================
# Setup: Import Libraries and Get Parameters
# ============================================================================

# Import PySpark functions for data transformation
from pyspark.sql.functions import col, when, monotonically_increasing_id, coalesce, first, concat, lit
from pyspark.sql.window import Window

In [0]:
# ============================================================================
# Step 1: Read Bronze Layer Data
# ============================================================================
# Load raw sales data from sales.bronze.retail_store_sales

sales_bronze_df = spark.read.table("sales.bronze.retail_store_sales")

In [0]:
# ============================================================================
# Step 2: Fast Cleaning - Remove Invalid Data & Standardize Types
# ============================================================================

# Remove rows where item is null (cannot have sales without product identification)
sales_silver_df = sales_bronze_df.filter(sales_bronze_df.item_id.isNotNull())

# Cast quantity from double to int (quantity should be whole numbers)
sales_silver_df = sales_silver_df.withColumn("quantity", col("quantity").cast("int"))

# Drop discount_applied column (not needed for silver layer analysis)
sales_silver_df = sales_silver_df.drop("discount_applied")

display(sales_silver_df)

In [0]:
# ============================================================================
# Step 3: Impute Missing Sales Metrics
# ============================================================================
# Use business logic to fill missing values based on the relationship:
# total_spent = price_per_unit * quantity
#
# - If price_per_unit is missing but total_spent and quantity exist: 
#   price_per_unit = total_spent / quantity
# - If quantity is missing but total_spent and price_per_unit exist:
#   quantity = total_spent / price_per_unit
# - If total_spent is missing but price_per_unit and quantity exist:
#   total_spent = price_per_unit * quantity
# ============================================================================

# Impute missing price_per_unit
sales_silver_df = sales_silver_df.withColumn(
    "price_per_unit",
    when(
        sales_silver_df.price_per_unit.isNull() & 
        sales_silver_df.total_spent.isNotNull() & 
        sales_silver_df.quantity.isNotNull(),
        sales_silver_df.total_spent / sales_silver_df.quantity
    ).otherwise(sales_silver_df.price_per_unit)
)

# Impute missing quantity
sales_silver_df = sales_silver_df.withColumn(
    "quantity",
    when(
        sales_silver_df.quantity.isNull() & 
        sales_silver_df.total_spent.isNotNull() & 
        sales_silver_df.price_per_unit.isNotNull(),
        (sales_silver_df.total_spent / sales_silver_df.price_per_unit).cast("int")
    ).otherwise(sales_silver_df.quantity)
)

# Impute missing total_spent
sales_silver_df = sales_silver_df.withColumn(
    "total_spent",
    when(
        sales_silver_df.total_spent.isNull() & 
        sales_silver_df.price_per_unit.isNotNull() & 
        sales_silver_df.quantity.isNotNull(),
        sales_silver_df.price_per_unit * sales_silver_df.quantity
    ).otherwise(sales_silver_df.total_spent)
)

# Filter out rows that still have null values after imputation
# (these are records where we couldn't derive missing values)
sales_silver_df = sales_silver_df.filter(
    sales_silver_df.total_spent.isNotNull() & 
    sales_silver_df.price_per_unit.isNotNull() & 
    sales_silver_df.quantity.isNotNull()
)

display(sales_silver_df)

In [0]:
# ============================================================================
# Data Quality Test: Verify No Null Values
# ============================================================================
# Test that all columns contain no null values after cleaning and imputation

from pyspark.sql.functions import sum as spark_sum

# Count null values in each column
null_counts = sales_silver_df.select(
    [spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
     for c in sales_silver_df.columns]
)

print("Null Value Counts by Column:")
display(null_counts)

# Get the null count results as a dictionary
null_count_dict = null_counts.collect()[0].asDict()

# Check if any column has null values
columns_with_nulls = {col_name: count for col_name, count in null_count_dict.items() if count > 0}

if columns_with_nulls:
    raise ValueError(f"Data quality test FAILED: Found null values in columns: {columns_with_nulls}")
else:
    print("\n✓ Data quality test PASSED: All columns contain no null values")

In [0]:
# ============================================================================
# Step 4: Write to Silver Layer
# ============================================================================
# Save the cleaned and validated data to the silver layer as a Delta table
# Target: sales.silver.retail_store_sales

sales_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales.silver.retail_store_sales")